# 5.3 — Partitions, Spill and Memory

**Chapter 5, sections 5.6 and 5.7**, and the starting point for **Exercises 2 and 4**.

**The question this notebook answers:** the chapter gives an executor's memory as an arithmetic
identity and gives the partition count as a four-step procedure, and the two are connected by a
single claim — that **more partitions, rather than more memory, is the first remedy for spill**.
This notebook computes the arithmetic and checks it against the running JVM, then provokes a real
spill and removes it by changing the partition count alone, with the memory configuration held
fixed.

It also builds, deliberately, the two stages of **Exercise 1**: one over-partitioned and one
skewed, so that the median-to-maximum comparison can be read on a stage where it actually says
something.

The driver is given a **1 GiB heap** on purpose. This notebook runs in `local[*]`, where the
driver JVM is also the executor, so `spark.executor.memory` would configure nothing; the heap
that matters is the driver's. Everything below is therefore about a small executor, which is what
makes the spill reproducible on a laptop.

Runs on a laptop in about a minute. The byte figures are exact; the timings are not.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, json, math, time, tempfile, logging, urllib.request
from urllib.parse import urlparse
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import spark_partition_id

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

DRIVER_HEAP_MIB = 1024        # deliberately small; see the note above

spark = (SparkSession.builder
         .appName("CS777-5.3")
         .master("local[*]")
         .config("spark.driver.memory", f"{DRIVER_HEAP_MIB}m")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
for _lg in ("SQLQueryContextLogger", "DataFrameQueryContextLogger"):
    logging.getLogger(_lg).setLevel(logging.CRITICAL)

pd.set_option("display.width", 200)

_port = urlparse(sc.uiWebUrl).port
UI = f"http://localhost:{_port}/api/v1"
APP = json.load(urllib.request.urlopen(f"{UI}/applications"))[0]["id"]

def ui(path):
    with urllib.request.urlopen(f"{UI}/applications/{APP}{path}", timeout=60) as r:
        return json.load(r)

print("Spark", spark.version)
print("driver heap asked for :", spark.conf.get("spark.driver.memory"))
print("driver heap the JVM has:",
      round(sc._jvm.java.lang.Runtime.getRuntime().maxMemory() / 1048576), "MiB")

Spark 4.2.0
driver heap asked for : 1024m
driver heap the JVM has: 1024 MiB


## 1. An executor's memory, as arithmetic

§5.6.1 states the division as a formula rather than a diagram, and the formula is short enough to
write as a function:

* about **300 MiB** is held back as a fixed reserve;
* `spark.memory.fraction` (0.6) of what remains is the unified region **$M$**, shared by
  execution and storage;
* `spark.memory.storageFraction` (0.5) of $M$ is the floor **$R$**, below which cached blocks
  cannot be evicted;
* the **container** the cluster manager must provide is the heap plus
  `memoryOverheadFactor` (0.10) times the heap — and in PySpark the Python workers are paid for
  out of that overhead, not out of the heap.

In [2]:
RESERVED_MIB = 300

def executor_budget(heap_mib,
                    memory_fraction=0.6,
                    storage_fraction=0.5,
                    overhead_factor=0.10,
                    min_overhead_mib=384):
    """The chapter's memory model, as arithmetic. All figures in MiB."""
    usable = heap_mib - RESERVED_MIB
    M = usable * memory_fraction
    R = M * storage_fraction
    overhead = max(heap_mib * overhead_factor, min_overhead_mib)
    return {"heap": heap_mib,
            "after the 300 MiB reserve": usable,
            "M = unified region": M,
            "R = eviction floor": R,
            "user memory (40%)": usable - M,
            "overhead": overhead,
            "container = heap + overhead": heap_mib + overhead}

# The chapter's own worked example, at line 357: an 8 GiB executor.
chapter = executor_budget(8 * 1024)
print("The chapter's 8 GiB executor\n")
for k, v in chapter.items():
    print(f"  {k:<28} {v:9.0f} MiB   ({v/1024:6.2f} GiB)")

print(f"\n  twenty such executors:")
print(f"    protected from eviction (20 x R) : {20*chapter['R = eviction floor']/1024:6.1f} GiB")
print(f"    available when nothing competes  : {20*chapter['M = unified region']/1024:6.1f} GiB")

The chapter's 8 GiB executor

  heap                              8192 MiB   (  8.00 GiB)
  after the 300 MiB reserve         7892 MiB   (  7.71 GiB)
  M = unified region                4735 MiB   (  4.62 GiB)
  R = eviction floor                2368 MiB   (  2.31 GiB)
  user memory (40%)                 3157 MiB   (  3.08 GiB)
  overhead                           819 MiB   (  0.80 GiB)
  container = heap + overhead       9011 MiB   (  8.80 GiB)

  twenty such executors:
    protected from eviction (20 x R) :   46.2 GiB
    available when nothing competes  :   92.5 GiB


### Checking the formula against the JVM

The formula is only worth trusting if Spark uses it. It does, and the Executors tab reports the
result: the `maxMemory` column **is** $M$. Below, $M$ is predicted from this notebook's own heap
and compared with what the running application reports.

In [3]:
predicted_M = executor_budget(DRIVER_HEAP_MIB)["M = unified region"]
reported_M = ui("/executors")[0]["maxMemory"] / 1048576

print(f"heap                          : {DRIVER_HEAP_MIB:8.1f} MiB")
print(f"M predicted by the formula    : {predicted_M:8.1f} MiB")
print(f"M reported by the Executors tab: {reported_M:8.1f} MiB")
print(f"difference                    : {abs(predicted_M - reported_M):8.2f} MiB")
assert abs(predicted_M - reported_M) < 1.0, "Spark's memory model no longer matches the chapter"
print("\nThe chapter's arithmetic is the arithmetic Spark is running.")

heap                          :   1024.0 MiB
M predicted by the formula    :    434.4 MiB
M reported by the Executors tab:    434.4 MiB
difference                    :     0.00 MiB

The chapter's arithmetic is the arithmetic Spark is running.


### Exercise 2, and the two definitions of the overhead factor

Exercise 2 asks for the same computation on a 12 GiB executor. It is one call.

The second table is §5.6.3's warning about vendors, and it is the sharper half. Apache defines
`memoryOverheadFactor` as a fraction of the **heap**; Amazon defines its overhead as a fraction of
the **container**. Applied to the same machine the two give different heaps, and neither is
wrong — they are answers to different questions.

In [4]:
ex2 = executor_budget(12 * 1024)
print("Exercise 2: a 12 GiB executor at the Apache defaults\n")
for k, v in ex2.items():
    print(f"  {k:<28} {v:9.0f} MiB   ({v/1024:6.2f} GiB)")

print("\n  (c) ten such executors offer"
      f" {10*ex2['R = eviction floor']/1024:.0f} GiB protected and"
      f" {10*ex2['M = unified region']/1024:.0f} GiB when nothing else competes,")
print("      so a 60 GiB dataset sits between the two: it fits only while nothing is computing,")
print("      and the Storage tab's 'fraction cached' is where that is checked rather than assumed.")

print("\n\n" + "=" * 78)
print("The AWS worked example of section 5.6.3, computed both ways\n")
MACHINE_MIB, MACHINE_CORES, CORES_PER_EXEC = 241_664, 32, 4
n_exec = MACHINE_CORES // CORES_PER_EXEC
container = MACHINE_MIB / n_exec
aws_heap = container * (1 - 0.1875)      # overhead is a fraction of the CONTAINER
apache_heap = container / 1.10           # overhead is a fraction of the HEAP

print(f"  machine offers          : {MACHINE_MIB:,} MiB over {MACHINE_CORES} cores")
print(f"  at {CORES_PER_EXEC} cores per executor : {n_exec} executors, container {container:,.0f} MiB each")
print(f"  Amazon, 0.1875 of the container -> heap {aws_heap:,.0f} MiB")
print(f"  Apache, 0.10 of the heap        -> heap {apache_heap:,.0f} MiB")
print(f"  difference                      : {apache_heap - aws_heap:,.0f} MiB per executor,"
      f" {n_exec*(apache_heap - aws_heap)/1024:,.1f} GiB per machine")

Exercise 2: a 12 GiB executor at the Apache defaults

  heap                             12288 MiB   ( 12.00 GiB)
  after the 300 MiB reserve        11988 MiB   ( 11.71 GiB)
  M = unified region                7193 MiB   (  7.02 GiB)
  R = eviction floor                3596 MiB   (  3.51 GiB)
  user memory (40%)                 4795 MiB   (  4.68 GiB)
  overhead                          1229 MiB   (  1.20 GiB)
  container = heap + overhead      13517 MiB   ( 13.20 GiB)

  (c) ten such executors offer 35 GiB protected and 70 GiB when nothing else competes,
      so a 60 GiB dataset sits between the two: it fits only while nothing is computing,
      and the Storage tab's 'fraction cached' is where that is checked rather than assumed.


The AWS worked example of section 5.6.3, computed both ways

  machine offers          : 241,664 MiB over 32 cores
  at 4 cores per executor : 8 executors, container 30,208 MiB each
  Amazon, 0.1875 of the container -> heap 24,544 MiB
  Apache, 0.10 of th

## 2. Slots, and the partition count as a procedure

§5.7.3 gives four steps, and the first is to count the slots: executors times cores. The two
vendor multipliers disagree, and the chapter's point is that neither multiplier is the target —
a **partition size** near 128 MB is.

In [5]:
slots = sc.defaultParallelism
print(f"slots on this machine            : {slots}")
print(f"shuffle partitions, as shipped   : {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"read-side target size            : "
      f"{int(spark.conf.get('spark.sql.files.maxPartitionBytes').rstrip('b'))/1024/1024:.0f} MiB")
print()

def partition_advice(slots_, shuffle_bytes):
    rules = [("the chapter, 2x slots", 2 * slots_),
             ("the chapter, 3x slots", 3 * slots_),
             ("Amazon, 1x vCores", 1 * slots_),
             ("Amazon, 2x vCores", 2 * slots_),
             ("Google, 3x vCPUs", 3 * slots_)]
    return pd.DataFrame([{"rule": name, "partitions": n,
                          "MB per partition": round(shuffle_bytes / n / 1e6, 1)}
                         for name, n in rules])

# Exercise 4: 12 executors of 4 cores, a stage that shuffles 24 GB.
EX4_SLOTS, EX4_SHUFFLE = 12 * 4, 24 * 1e9
print(f"Exercise 4: {EX4_SLOTS} slots, a stage shuffling {EX4_SHUFFLE/1e9:.0f} GB\n")
print(partition_advice(EX4_SLOTS, EX4_SHUFFLE).to_string(index=False))
print(f"\n  To land on 128 MB per partition exactly:"
      f" {EX4_SHUFFLE/128e6:.0f} partitions, which is"
      f" {EX4_SHUFFLE/128e6/EX4_SLOTS:.1f}x the slot count --")
print("  so on this cluster shape every rule above under-partitions, and the size is the")
print("  quantity that says so. The multiplier is a convenience, not the target.")

slots on this machine            : 18
shuffle partitions, as shipped   : 200
read-side target size            : 128 MiB

Exercise 4: 48 slots, a stage shuffling 24 GB

                 rule  partitions  MB per partition
the chapter, 2x slots          96             250.0
the chapter, 3x slots         144             166.7
    Amazon, 1x vCores          48             500.0
    Amazon, 2x vCores          96             250.0
     Google, 3x vCPUs         144             166.7

  To land on 128 MB per partition exactly: 188 partitions, which is 3.9x the slot count --
  so on this cluster shape every rule above under-partitions, and the size is the
  quantity that says so. The multiplier is a convenience, not the target.


## 3. Seeing the distribution rather than assuming it

The chapter's listing at §5.7.3 gives two ways to look at the actual contents of the partitions:
`glom` on an RDD and a grouping by `spark_partition_id` on a DataFrame. Both are diagnostics on a
sample, because both return one value per partition to the driver.

Two datasets are built below with a fixed seed: one balanced, and one in which a single key holds
80 % of the records, which is the shape Exercise 10 of the chapter asks for.

In [6]:
N = 4_000_000

balanced = (spark.range(0, N)
            .withColumn("k", (F.rand(seed=7) * 1_000_000).cast("long"))
            .withColumn("pad", F.expr("repeat('x', 120)")))

# One key in every five records is the same key, and the rest are spread over a million others.
skewed = (spark.range(0, N)
          .withColumn("k", F.when(F.rand(seed=11) < 0.8, F.lit(0))
                            .otherwise((F.rand(seed=13) * 1_000_000).cast("long") + 1))
          .withColumn("pad", F.expr("repeat('x', 120)")))

def per_partition(df, label, top=5):
    counts = (df.withColumn("pid", spark_partition_id())
                .groupBy("pid").count()
                .orderBy("count", ascending=False)
                .limit(top).toPandas())
    print(f"{label}: the {top} largest partitions")
    print(counts.to_string(index=False))

# The RDD form of the same question, from the chapter's listing.
rdd = sc.parallelize(range(1, 15), 5)
print("rdd.getNumPartitions() ->", rdd.getNumPartitions())
print("rdd.glom().map(len).collect() ->", rdd.glom().map(len).collect())
print()

spark.conf.set("spark.sql.shuffle.partitions", 32)
per_partition(balanced.repartition(32, "k"), "balanced, hashed into 32 partitions")
print()
per_partition(skewed.repartition(32, "k"), "80 % on one key, hashed into 32 partitions")

rdd.getNumPartitions() -> 5


rdd.glom().map(len).collect() -> [2, 3, 3, 3, 3]



balanced, hashed into 32 partitions: the 5 largest partitions
 pid  count
  13 127784
   4 126944
  30 126380
   2 126046
  11 125946



80 % on one key, hashed into 32 partitions: the 5 largest partitions
 pid   count
  29 3223864
  13   25503
  23   25385
   4   25381
  11   25195


The second table is the whole of data skew in one picture. The partitioning is correct — every
record went to the partition its key hashes to — and one partition holds four fifths of the data
anyway, because four fifths of the records carry the same key. No partition count repairs this,
which is the chapter's warning at step 4 of the procedure.

## 4. Spill, and the remedy that is not more memory

§5.6.2 says any nonzero spill means execution memory ran short, and that of the three available
responses, more partitions is usually the better one. That is a testable claim: hold the memory
configuration fixed, change only the partition count, and watch the spill rows.

Adaptive Query Execution is switched off for this measurement, and the reason is worth stating:
AQE re-decides the post-shuffle partition count at run time from the configured value, so leaving
it on would measure AQE rather than the setting. It is switched back on in the next section, where
that behaviour is the subject.

In [7]:
spark.conf.set("spark.sql.adaptive.enabled", "false")

def reduce_stage(stages):
    """The stage on the READ side of the shuffle -- the one the partition count governs.

    Picking the longest-running stage is wrong here: on a laptop the scan is often the longest,
    and it has nothing to do with the shuffle width being varied."""
    reading = [s for s in stages if s["shuffleReadBytes"] > 0]
    return max(reading or stages, key=lambda s: s["executorRunTime"])

def run_and_measure(label, n_partitions, df):
    """Run a global sort at a given shuffle width; report its spill and task distribution."""
    spark.conf.set("spark.sql.shuffle.partitions", n_partitions)
    sc.setJobDescription(label)
    before = {s["stageId"] for s in ui("/stages")}
    t0 = time.time()
    df.orderBy("k").write.mode("overwrite").format("noop").save()
    secs = time.time() - t0
    new = [s for s in ui("/stages") if s["stageId"] not in before]
    worst = reduce_stage(new)
    d = ui(f"/stages/{worst['stageId']}/0/taskSummary?quantiles=0.5,1.0")
    concurrent = min(worst["numTasks"], slots)
    return {"shuffle.partitions": n_partitions,
            "tasks": worst["numTasks"],
            "rows per task": f"{N // max(worst['numTasks'], 1):,}",
            "concurrent": concurrent,
            "M per running task MiB": round(reported_M / concurrent),
            "spill, memory MB": round(sum(s["memoryBytesSpilled"] for s in new) / 1e6, 1),
            "spill, disk MB": round(sum(s["diskBytesSpilled"] for s in new) / 1e6, 1),
            "median task ms": round(d["duration"][0]),
            "max task ms": round(d["duration"][1])}

print(f"{N:,} rows, sorted globally. Memory configuration held fixed at a"
      f" {DRIVER_HEAP_MIB} MiB heap, M = {reported_M:.0f} MiB.\n")
results = [run_and_measure(f"sort at {n} partitions", n, balanced) for n in (2, 8, 32, 128)]
print(pd.DataFrame(results).to_string(index=False))

first, last = results[0], results[-1]
print(f"\nSpill fell from {first['spill, memory MB']:,.0f} MB to"
      f" {last['spill, memory MB']:,.0f} MB without one byte of extra memory.")
if first["spill, disk MB"]:
    print(f"The memory figure runs about {first['spill, memory MB']/first['spill, disk MB']:.0f}x"
          " the disk figure, which is the deserialized-to-serialized ratio the chapter")
    print("describes, not a defect: the same records occupy far more space as JVM objects.")

4,000,000 rows, sorted globally. Memory configuration held fixed at a 1024 MiB heap, M = 434 MiB.



 shuffle.partitions  tasks rows per task  concurrent  M per running task MiB  spill, memory MB  spill, disk MB  median task ms  max task ms
                  2      2     2,000,000           2                     217             329.2            27.5             638          638
                  8      8       500,000           8                      54             419.4            34.5             257          278
                 32     32       125,000          18                      24              83.9             6.2              58           76
                128    128        31,250          18                      24               0.0             0.0              14           29

Spill fell from 329 MB to 0 MB without one byte of extra memory.
The memory figure runs about 12x the disk figure, which is the deserialized-to-serialized ratio the chapter
describes, not a defect: the same records occupy far more space as JVM objects.


### The descent is not monotonic, and the extra column says why

Read the table with the `concurrent` column beside the spill. Raising the partition count does
two things at once, and only one of them helps. Each task is handed **less data**, which is the
effect being sought. But more partitions also means **more tasks running at the same time**, up
to the slot count, and execution memory is shared among whatever is running — so the budget per
task falls at the same time as the demand per task does.

Between the first two rows the second effect wins and the spill goes *up*. Past that point the
slot count caps the concurrency while the data per task keeps falling, and the spill collapses to
nothing. The chapter's rule survives, with its mechanism made explicit: what removes spill is a
smaller working set **relative to the share of $M$ the task will actually get**, and on a machine
with more slots than partitions the two move together.

## 5. The two readings of Exercise 1, built deliberately

Exercise 1 gives two stages and asks for a diagnosis of each. Both are constructed below, and
both are read off the same summary-metrics table, so that the comparison the chapter calls the
median-to-maximum comparison is made on stages that genuinely differ.

In [8]:
# A dimension table to join against, so that a stage exists which cannot combine on the map side.
dim = (spark.range(0, 1_000_001)
       .withColumnRenamed("id", "k")
       .withColumn("label", F.concat(F.lit("key-"), F.col("k").cast("string"))))

def diagnose(label, n_partitions, df, kind="agg", adaptive=False, broadcast=False):
    spark.conf.set("spark.sql.adaptive.enabled", "true" if adaptive else "false")
    spark.conf.set("spark.sql.shuffle.partitions", n_partitions)
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1 if not broadcast else 10485760)
    sc.setJobDescription(label)
    before = {s["stageId"] for s in ui("/stages")}
    if kind == "agg":
        df.groupBy("k").count().write.mode("overwrite").format("noop").save()
    else:
        # A little real work per row, so that an imbalance in records becomes an imbalance
        # in seconds. Without it the hot task reads 50x the records and finishes almost as
        # quickly, because writing to noop costs nothing per row.
        (df.join(dim, "k")
           .select(F.sha2(F.concat_ws("|", "label", "pad"), 512).alias("h"))
           .write.mode("overwrite").format("noop").save())
    new = [s for s in ui("/stages") if s["stageId"] not in before]
    worst = reduce_stage(new)
    d = ui(f"/stages/{worst['stageId']}/0/taskSummary?quantiles=0,0.5,1.0")
    median, mx = d["duration"][1], d["duration"][2]
    read = d["shuffleReadMetrics"]["readRecords"]
    ratio = mx / max(median, 1)
    row_ratio = read[2] / max(read[1], 1)
    if ratio >= 3 and mx >= 100:
        verdict = f"skew: the slowest task is {ratio:.0f}x the median"
    elif row_ratio >= 3:
        verdict = f"skew in the data ({row_ratio:.0f}x the records), not yet in the clock"
    elif median < 50 and worst["numTasks"] > 4 * slots:
        verdict = "over-partitioned: tasks too short to pay for themselves"
    else:
        verdict = "healthy: balanced, substantial tasks"
    return {"stage": label, "tasks": worst["numTasks"],
            "median ms": round(median), "max ms": round(mx),
            "max/median": round(ratio, 1),
            "median rows read": f"{read[1]:,.0f}", "max rows read": f"{read[2]:,.0f}",
            "rows max/median": round(row_ratio, 1),
            "diagnosis": verdict}

print("First, the obvious probe: group the skewed data by its key and count.\n")
print(pd.DataFrame([
    diagnose("32 partitions, 80% on one key, groupBy+count", 32, skewed, kind="agg"),
]).to_string(index=False))

First, the obvious probe: group the skewed data by its key and count.



                                       stage  tasks  median ms  max ms  max/median median rows read max rows read  rows max/median                            diagnosis
32 partitions, 80% on one key, groupBy+count     32         67      84         1.3           24,497        24,963              1.0 healthy: balanced, substantial tasks


**The skew has vanished, and that is not an error in the measurement.** `count` combines on the
map side: each input partition reduces its three million records for the hot key to a single
partial count *before* the shuffle, so the reduce task that owns that key receives one row from
each upstream task rather than three million records. This is the `reduceByKey` mechanism of
§5.10.2 arriving in the structured API, where it is automatic and invisible.

It also means the obvious probe is the wrong instrument. To see skew in a stage, the stage has to
be one that cannot combine — a join, for instance, where every record on the skewed side must
physically arrive at the task that owns its key.

In [9]:
rows = [
    diagnose("a) 2,000 partitions, balanced, groupBy+count", 2000, balanced, kind="agg"),
    diagnose("b) 32 partitions, 80% on one key, JOIN",        32, skewed,   kind="join"),
    diagnose("c) 32 partitions, keys spread evenly, JOIN",    32, balanced, kind="join"),
]
print(pd.DataFrame(rows).to_string(index=False))

                                       stage  tasks  median ms  max ms  max/median median rows read max rows read  rows max/median                                               diagnosis
a) 2,000 partitions, balanced, groupBy+count   2000          6      50         8.3            1,794         2,079              1.2 over-partitioned: tasks too short to pay for themselves
      b) 32 partitions, 80% on one key, JOIN     32        215    1227         5.7           56,263     3,255,232             57.9                 skew: the slowest task is 6x the median
  c) 32 partitions, keys spread evenly, JOIN     32         97     141         1.5          156,133       159,484              1.0                    healthy: balanced, substantial tasks


Row (a) is the over-partitioned stage: thousands of tasks, each one over almost immediately, and
the stage's time going on dispatching them rather than on computing. Row (b) is skew, and the two
rows-read columns are the clearest statement of it: the median task reads a few tens of thousands
of records and the maximum reads millions, from the same stage, under the same partition count.
Row (c) is the control, and it matters — without it, any stage can be made to look like a
pathology.

There is a caution in row (b) worth carrying to a real cluster. The chapter's test is on the
**duration** row, and duration is a derived quantity: it is records multiplied by the work each
record costs. Remove the work — write the joined rows straight to a sink that does nothing with
them — and the same stage reads fifty times the records in the hot task and finishes in barely
twice the time, because reading records it does nothing with is nearly free. The records columns
show the imbalance in either case. **Shuffle read records is the more sensitive instrument, and
duration is the one that matters**; when they disagree, the imbalance is real and is waiting for
a workload heavy enough to reveal it.

**The remedies point in opposite directions**, which is why the chapter insists the comparison is
made before anything is changed. Row (a) wants *fewer* partitions. Row (b) would be made worse by
fewer, and is not repaired by more either: the oversized partition is one key, and a key cannot be
divided by a partition count.

### What Adaptive Query Execution does to row (a)

§5.7.3 closes with the qualification that AQE does not make the partition count irrelevant; it
moves the decision from planning time to run time, working from the configured count as a
starting point. The same over-partitioned stage is run again below with AQE on.

In [10]:
aqe_off = diagnose("2,000 partitions, AQE off", 2000, balanced, adaptive=False)
aqe_on  = diagnose("2,000 partitions, AQE on",  2000, balanced, adaptive=True)
print(pd.DataFrame([aqe_off, aqe_on]).to_string(index=False))

print(f"\nThe configured number was 2,000 in both rows. AQE coalesced it to"
      f" {aqe_on['tasks']} after seeing the true partition sizes.")
print("What it cannot revise is the read-side count, which is fixed before any statistics exist;")
print("that one is the programmer's, and notebook 5.1 is where it is measured.")
spark.conf.set("spark.sql.adaptive.enabled", "true")

                    stage  tasks  median ms  max ms  max/median median rows read max rows read  rows max/median                                               diagnosis
2,000 partitions, AQE off   2000          6      49         8.2            1,794         2,079              1.2 over-partitioned: tasks too short to pay for themselves
 2,000 partitions, AQE on     18         95      96         1.0          198,487       215,063              1.1                    healthy: balanced, substantial tasks

The configured number was 2,000 in both rows. AQE coalesced it to 18 after seeing the true partition sizes.
What it cannot revise is the read-side count, which is fixed before any statistics exist;
that one is the programmer's, and notebook 5.1 is where it is measured.


## Conclusion

The memory model and the partition count are one subject, because the partition count is the
cheaper of the two controls over the same quantity: how much data one task must hold at once.

What this notebook establishes by running it:

1. **The chapter's memory arithmetic is Spark's.** $M$ predicted from the heap agreed with the
   `maxMemory` the Executors tab reports, to within a rounding error, and the assertion in
   section 1 will fail loudly if a future release changes the model.
2. **Spill is removed by partitioning, with the memory configuration untouched.** Four runs over
   the same data at four partition counts took the spill from hundreds of megabytes to zero. The
   memory figure ran an order of magnitude above the disk figure throughout, which is the
   deserialized-to-serialized ratio and not a fault.
3. **The median-to-maximum comparison distinguishes two pathologies that want opposite
   remedies**, and a third stage that is simply healthy. A diagnostic that only ever fires is not
   a diagnostic.
4. **Skew is not a partitioning error.** Every record went where its key hashes to, and one
   partition still held four fifths of the data. The remedy is in the key, not in the count.
5. **AQE revises the shuffle-side count and not the read-side count.** It turned 2,000 configured
   partitions into a handful at run time; the number of partitions a file read produces is decided
   before any statistic exists.

*Chapter sections:* §5.6 (memory configuration), §5.7 (maximizing parallelism).
*Exercise 1* is answered by the table in section 5, *Exercise 2* by the two budgets in section 1,
and *Exercise 4* by `partition_advice` in section 2 together with the skew row of section 5.